# 직장 결근 시간 예측 — EDA와 모델 학습 (Absenteeism)

- 데이터: 브라질 택배회사 직원 36명의 결근 기록 (740행 · 21열)
- 목표: 결근 시간(Absenteeism time in hours) 예측 — 회귀
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_Absenteeism.txt

- 이 노트북은 따릉이 실습과 같은 흐름을 다른 데이터로 재현한다

## 1. 불러오기

- 이 파일은 구분자가 **세미콜론(;)** → sep 지정 필수
- 그냥 read_csv 하면 전체가 1열이 되어 학습 불가

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Absenteeism_at_work.csv", sep=";")
print(df.shape)          # (740, 21)
df.head()

### 1-1. 컬럼명 정리

- 'Work load Average/day ' 는 끝에 공백·슬래시가 있음
- label 지정·컬럼 참조 시 오타를 유발하므로 정리

In [ ]:
df.columns = (df.columns
                .str.strip()
                .str.replace("/", "_", regex=False)
                .str.replace(" ", "_", regex=False))
print(df.columns.tolist())

## 2. 학습 전 확인 — AutoGluon이 잘못 학습할 것 바로잡기

- 결측·인코딩은 AutoGluon이 자동 처리
- 사람이 확인할 것: 숫자로 위장한 범주 · 같은 직원 중복

### 2-1. 숫자로 코딩된 범주 (가장 중요)

- Reason(사유)·Month·Day·Seasons는 숫자지만 **범주**다
- 그냥 두면 "28번 사유가 1번의 28배"로 잘못 학습
- 범주형으로 지정해 순서·크기 의미를 제거

In [ ]:
cat = ["Reason_for_absence", "Month_of_absence",
       "Day_of_the_week", "Seasons"]
for c in cat:
    df[c] = df[c].astype("category")

print(df[cat].dtypes)
# 0/1 이진 변수(Disciplinary_failure 등)는 AutoGluon이 자동 인식

### 2-2. 같은 직원이 여러 번 등장

- 직원 36명이 740행에 반복 등장 (한 명이 여러 번 결근)
- 무작위 분할 시 같은 직원이 train·test 양쪽에 → 성능 과대평가
- ID 단위로 분할해야 '새 직원'에 대한 성능을 정직하게 측정

In [ ]:
# 확인: 직원 수 vs 행 수
print("직원 수:", df["ID"].nunique(), "/ 행 수:", len(df))

# ID 단위 분할
ids = df["ID"].unique()
rng = np.random.RandomState(42)
train_ids = rng.choice(ids, size=int(len(ids)*0.8), replace=False)

train_data = df[df["ID"].isin(train_ids)].copy()
test_data  = df[~df["ID"].isin(train_ids)].copy()
print("train:", train_data.shape, "test:", test_data.shape)

## 3. EDA

- ProfileReport로 분포·결측·상관을 한 번에 확인
- 이 데이터는 결측이 없음 → 분포와 관계에 집중

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, progress_bar=False)
profile.to_file("absenteeism_eda.html")   # 브라우저에서 열기

### 3-1. target 살펴보기

- 결근 시간(target)의 분포 확인
- 0인 행이 존재 → 실제 결근이 아닌 기록일 수 있음
- 한쪽으로 치우친 분포 → 로그 변환 검토

In [ ]:
t = "Absenteeism_time_in_hours"
print(df[t].describe())
print("\n0인 행:", (df[t]==0).sum(), f"({(df[t]==0).mean():.1%})")

df[t].hist(bins=40, figsize=(8,3))
import matplotlib.pyplot as plt
plt.title("distribution of absenteeism hours")
plt.show()

### 3-2. 결근 사유별 평균 결근 시간

- 어떤 사유가 결근을 길게 만드는가
- 범주형으로 본 관계 (해석용)

In [ ]:
(df.groupby("Reason_for_absence", observed=True)[t]
   .mean().sort_values(ascending=False).head(10)
   .plot(kind="bar", figsize=(10,3), title="mean hours by reason"))
plt.show()

## 4. 학습

- TabularPredictor 한 줄로 여러 모델을 학습·앙상블
- label = 예측 대상, eval_metric = 평가 지표
- 결측·인코딩·범주 처리는 내부에서 자동

In [ ]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    label="Absenteeism_time_in_hours",
    eval_metric="root_mean_squared_error",
).fit(
    train_data,
    presets="medium_quality",
    time_limit=300,
)

### 4-1. 리더보드 — 모델 비교

- AutoGluon이 학습한 모델들의 성능 순위
- 맨 위가 최종 앙상블(WeightedEnsemble) — 보통 가장 좋음

In [ ]:
predictor.leaderboard(test_data)

## 5. 해석

- 예측 성능 확인 + 어떤 변수가 중요했는지 확인

### 5-1. 테스트 성능

In [ ]:
perf = predictor.evaluate(test_data)
print(perf)

### 5-2. 변수 중요도

- 어떤 변수가 예측에 크게 기여했는가
- 결과가 상식과 맞는지 판단 (예: 결근 사유가 중요한가?)

In [ ]:
predictor.feature_importance(test_data)

### 5-3. 예측 확인

- 실제값과 예측값을 나란히 비교

In [ ]:
pred = predictor.predict(test_data)
compare = pd.DataFrame({
    "actual": test_data["Absenteeism_time_in_hours"].values,
    "predicted": pred.values,
}).head(15)
print(compare)

## 정리

- 세미콜론 구분자·컬럼명 공백 → 불러오기 단계에서 처리
- 숫자로 위장한 범주 → 범주형 지정 (사람의 판단)
- 같은 직원 중복 → ID 단위 분할 (누수 방지)
- 학습·앙상블은 TabularPredictor가 자동
- 변수 중요도로 결과를 해석 → "핵심은 사람"

- 다른 데이터로도 같은 흐름을 반복해 볼 것 (data_*.txt 참조)